# Σημειώσεις Pandas: Πίνακες Διπλής Εισόδου (`pd.crosstab()`)

Η συνάρτηση `pd.crosstab()` δημιουργεί έναν **πίνακα διπλής εισόδου** (cross-tabulation ή contingency table), παρόμοιο με τα **Pivot Tables** του Excel. Είναι ο πιο γρήγορος τρόπος για να μετρήσουμε τη συχνότητα εμφάνισης συνδυασμών ανάμεσα σε **δύο κατηγορικές μεταβλητές**.

---

## 1. Βασική Σύνταξη (Καταμέτρηση Συχνοτήτων)

Από προεπιλογή (default), η `crosstab()` μετράει πόσες φορές εμφανίζεται κάθε συνδυασμός γραμμής και στήλης.

```python
import pandas as pd

# Σύνταξη: pd.crosstab(index=γραμμές, columns=στήλες)
pd.crosstab(retail["region"], retail["channel"])
```
* **1ο όρισμα (`index`):** Ορίζει ποιες τιμές θα γίνουν οι **γραμμές** του νέου πίνακα.
* **2ο όρισμα (`columns`):** Ορίζει ποιες τιμές θα γίνουν οι **στήλες** του νέου πίνακα.

---

## 2. Χρήσιμες Παράμετροι για Ανάλυση

### Α. Προσθήκη Γενικών Συνόλων (`margins=True`)
Προσθέτει μια επιπλέον γραμμή και μια επιπλέον στήλη (`All`) με τα αθροίσματα ανά γραμμή, ανά στήλη, καθώς και το γενικό σύνολο.

```python
pd.crosstab(retail["region"], retail["channel"], margins=True)
```

### Β. Ποσοστά αντί για Απόλυτους Αριθμούς (`normalize`)
Ιδανικό για **behavioral analysis**, καθώς δείχνει την κατανομή σε ποσοστά (από 0 έως 1).
* `normalize="all"` (ή `True`): Ποσοστό επί του **γενικού συνόλου** του πίνακα.
* `normalize="index"`: Ποσοστό **ανά γραμμή** (κάθε γραμμή αθροίζει στο 1.0 / 100%).
* `normalize="columns"`: Ποσοστό **ανά στήλη** (κάθε στήλη αθροίζει στο 1.0 / 100%).

```python
# Ποσοστό καναλιών πώλησης μέσα σε κάθε περιοχή (πολλαπλασιάζουμε με 100 για %)
pd.crosstab(retail["region"], retail["channel"], normalize="index").round(4) * 100
```

### Γ. Υπολογισμός Άλλης Μεταβλητής (`values` & `aggfunc`)
Αντί για απλό μέτρημα παραγγελιών, μπορούμε να υπολογίσουμε το άθροισμα ή τον μέσο όρο μιας τρίτης αριθμητικής στήλης.

```python
# Συνολικά έσοδα (revenue) ανά περιοχή και κανάλι
pd.crosstab(
    index=retail["region"], 
    columns=retail["channel"], 
    values=retail["revenue"], 
    aggfunc="sum"
)
```

---

## 💡 Σύνοψη: Γιατί να την προτιμήσω;
* Αντί να κάνεις `retail.groupby(["region", "channel"]).size().unstack()`, η `pd.crosstab()` το κάνει **σε μία μόνο γραμμή κώδικα**.
* Αποδέχεται απευθείας Series (στήλες) και είναι εξαιρετικά ευανάγνωστη για γρήγορα reports.


In [1]:
import pandas as pd

In [2]:
retail = pd.read_csv("../data/retail.csv", index_col=0)

In [3]:
retail.head(3)

,date,category,product,region,channel,units,unit_price,discount_pct,revenue
order_id,,,,,,,,,
ORD-10000,2025-10-15,Books,Puzzle book,East,store,1,178.5,15,178.5
ORD-10001,2025-11-07,Home,Storage box,East,app,1,663.0,15,663.0
ORD-10002,2025-12-26,Stationery,Gel pen pack,East,web,1,90.0,0,90.0


In [4]:
pd.crosstab(retail['region'], retail['channel'])

channel,app,store,web
region,,,
East,126,94,146
North,126,100,117
South,143,86,128
West,128,80,126


In [5]:
pd.crosstab(retail['category'], retail['channel']).sort_values('category')

channel,app,store,web
category,,,
Books,75,70,210
Electronics,227,63,74
Home,117,119,104
Stationery,104,108,129


In [6]:
ct = pd.crosstab(retail['category'], retail['region'], margins=True) #margins adds total sum
ct

region,East,North,South,West,All
category,,,,,
Books,99,79,94,83,355
Electronics,93,100,75,96,364
Home,85,72,96,87,340
Stationery,89,92,92,68,341
All,366,343,357,334,1400


In [7]:
ct.iloc[0,0:2]

region
East     99
North    79
Name: Books, dtype: int64

In [8]:
ct.loc['Home']

region
East      85
North     72
South     96
West      87
All      340
Name: Home, dtype: int64

In [9]:
(pd.crosstab(retail['category'], retail['channel'], normalize='index')*100).round(2) #ποσοστά ανα γραμμή (index)

channel,app,store,web
category,,,
Books,21.13,19.72,59.15
Electronics,62.36,17.31,20.33
Home,34.41,35.00,30.59
Stationery,30.50,31.67,37.83


In [10]:
(pd.crosstab(retail['category'],retail['channel'],normalize='columns')*100).round(2)

channel,app,store,web
category,,,
Books,14.34,19.44,40.62
Electronics,43.40,17.50,14.31
Home,22.37,33.06,20.12
Stationery,19.89,30.00,24.95


In [11]:
pd.crosstab(index=retail['category'],
            columns=retail['channel'],
            values=retail['revenue'],
            aggfunc='sum',
            margins=True)

channel,app,store,web,All
category,,,,
Books,80572.05,57515.4,190617.45,328704.9
Electronics,636135.00,177450.0,196432.50,1010017.5
Home,216188.50,240374.5,181087.00,637650.0
Stationery,18586.50,19363.5,22585.50,60535.5
All,951482.05,494703.4,590722.45,2036907.9


In [12]:
retail.groupby('channel')['revenue'].sum().reset_index(name='sum').sort_values('channel')

,channel,sum
0,app,951482.05
1,store,494703.40
2,web,590722.45


In [13]:
retail.groupby(by=['category','channel'])['revenue'].sum().reset_index(name='sum')

,category,channel,sum
0,Books,app,80572.05
1,Books,store,57515.40
2,Books,web,190617.45
3,Electronics,app,636135.00
4,Electronics,store,177450.00
5,Electronics,web,196432.50
6,Home,app,216188.50
7,Home,store,240374.50
8,Home,web,181087.00
9,Stationery,app,18586.50


In [14]:
pd.crosstab(index=retail['category'],
            columns=retail['channel'],
            values=retail['revenue'],
            aggfunc='mean',
            normalize='index').round(2)*100

channel,app,store,web
category,,,
Books,38.0,29.0,32.0
Electronics,34.0,34.0,32.0
Home,33.0,36.0,31.0
Stationery,34.0,34.0,33.0


In [20]:
pd.crosstab(index=retail['category'],
            columns=retail['channel'],
            values=retail['revenue'],
            aggfunc='median'
            )

channel,app,store,web
category,,,
Books,798.0,693.75,712.5
Electronics,2160.0,2160.00,2040.0
Home,1404.0,1560.00,1402.5
Stationery,136.5,120.00,120.0


In [26]:
pd.crosstab(retail['region'],
            [retail['category'],retail['channel']],
            margins='all'
            )

category Books            Electronics           Home            Stationery  \
channel    app store  web         app store web  app store  web        app   
region                                                                       
East        20    20   59          52    16  25   26    32   27         28   
North       18    17   44          64    19  17   21    28   23         23   
South       18    22   54          52     9  14   40    32   24         33   
West        19    11   53          59    19  18   30    27   30         20   
All         75    70  210         227    63  74  117   119  104        104   

category              All  
channel  store  web        
region                     
East        26   35   366  
North       36   33   343  
South       23   36   357  
West        23   25   334  
All        108  129  1400